# Logistic Regression — from scratch

Logistic regression is a **classification** model. It passes a linear score through the sigmoid to get a probability:

$$P(y=1\mid\mathbf{x}) = \sigma(\mathbf{w}^\top\mathbf{x} + b), \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$

Parameters are found by minimising the **binary cross-entropy** (negative log-likelihood), optionally with an L2 penalty:

$$J = -\frac{1}{n}\sum_i \big[y_i\log p_i + (1-y_i)\log(1-p_i)\big] + \frac{\lambda}{2n}\lVert\mathbf{w}\rVert^2$$

The gradient has the same form as linear regression:

$$\nabla_{\mathbf{w}} J = \frac{1}{n}X^\top(\mathbf{p} - \mathbf{y}) + \frac{\lambda}{n}\mathbf{w}, \qquad \frac{\partial J}{\partial b} = \frac{1}{n}\sum_i (p_i - y_i)$$

For $K>2$ classes we generalise to **softmax regression** (multinomial logistic regression).

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

In [ ]:
def make_blobs(n_samples=300, centers=3, cluster_std=1.0, n_features=2,
               center_box=(-10, 10), random_state=None):
    """Isotropic Gaussian blobs (re-implementation of sklearn's make_blobs)."""
    rng = np.random.RandomState(random_state)
    if isinstance(centers, int):
        centers = rng.uniform(*center_box, size=(centers, n_features))
    centers = np.asarray(centers, dtype=float)
    k = len(centers)
    counts = [n_samples // k + (1 if i < n_samples % k else 0) for i in range(k)]
    stds = np.broadcast_to(np.asarray(cluster_std, dtype=float), (k,))
    X = np.vstack([rng.normal(c, s, size=(m, centers.shape[1]))
                   for c, s, m in zip(centers, stds, counts)])
    y = np.concatenate([np.full(m, i) for i, m in enumerate(counts)])
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def train_test_split(X, y, test_size=0.25, random_state=None):
    """Shuffle and split arrays into train and test subsets."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(X))
    n_test = int(round(len(X) * test_size))
    test, train = idx[:n_test], idx[n_test:]
    return X[train], X[test], y[train], y[test]

def accuracy_score(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

def plot_decision_boundary(model, X, y, title="", ax=None, resolution=150):
    """Colour the plane by the model's prediction and overlay the data."""
    ax = ax or plt.gca()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    Z = np.asarray(model.predict(np.c_[xx.ravel(), yy.ravel()]), dtype=float).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
    ax.set_title(title)

## Binary logistic regression

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))


class LogisticRegression:
    def __init__(self, lr=0.1, n_iters=1000, l2=0.0):
        self.lr = lr
        self.n_iters = n_iters
        self.l2 = l2

    def fit(self, X, y):
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        self.loss_history = []
        eps = 1e-12
        for _ in range(self.n_iters):
            p = sigmoid(X @ self.w + self.b)
            loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
            loss += self.l2 / (2 * n) * np.sum(self.w ** 2)
            self.loss_history.append(loss)

            dw = X.T @ (p - y) / n + self.l2 / n * self.w
            db = np.mean(p - y)
            self.w -= self.lr * dw
            self.b -= self.lr * db
        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

In [ ]:
X, y = make_blobs(n_samples=400, centers=[[-2, -1], [2, 1.5]], cluster_std=1.5, random_state=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

clf = LogisticRegression(lr=0.1, n_iters=2000).fit(X_train, y_train)
print("weights:", clf.w, " bias:", clf.b)
print("Train accuracy:", accuracy_score(y_train, clf.predict(X_train)))
print("Test accuracy :", accuracy_score(y_test, clf.predict(X_test)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_decision_boundary(clf, X_test, y_test, "Decision boundary (test set)", ax=axes[0])
# the boundary w.x + b = 0 as a line
xs = np.linspace(X[:, 0].min(), X[:, 0].max(), 10)
axes[0].plot(xs, -(clf.w[0] * xs + clf.b) / clf.w[1], "k--")

axes[1].plot(clf.loss_history)
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("cross-entropy")
axes[1].set_title("Training loss")
plt.tight_layout(); plt.show()

### The sigmoid and predicted probabilities

In [ ]:
z = np.linspace(-8, 8, 200)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(z, sigmoid(z)); axes[0].axhline(0.5, ls="--", c="gray"); axes[0].set_title("sigmoid(z)")

xx, yy = np.meshgrid(np.linspace(-7, 7, 200), np.linspace(-6, 7, 200))
P = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
cs = axes[1].contourf(xx, yy, P, levels=20, cmap="RdBu_r", alpha=0.8)
plt.colorbar(cs, ax=axes[1], label="P(y=1)")
axes[1].scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=15)
axes[1].set_title("Predicted probability surface")
plt.tight_layout(); plt.show()

## Multiclass: softmax regression

$$P(y=k\mid\mathbf{x}) = \frac{e^{\mathbf{w}_k^\top\mathbf{x}+b_k}}{\sum_j e^{\mathbf{w}_j^\top\mathbf{x}+b_j}}$$

With one-hot labels $Y$, the cross-entropy gradient is $\frac{1}{n}X^\top(P - Y)$.

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)      # numerical stability
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)


class SoftmaxRegression:
    def __init__(self, lr=0.1, n_iters=1000, l2=0.0):
        self.lr, self.n_iters, self.l2 = lr, n_iters, l2

    def fit(self, X, y):
        n, d = X.shape
        self.classes_ = np.unique(y)
        k = len(self.classes_)
        Y = np.eye(k)[np.searchsorted(self.classes_, y)]   # one-hot
        self.W = np.zeros((d, k))
        self.b = np.zeros(k)
        self.loss_history = []
        for _ in range(self.n_iters):
            P = softmax(X @ self.W + self.b)
            self.loss_history.append(-np.mean(np.sum(Y * np.log(P + 1e-12), axis=1)))
            self.W -= self.lr * (X.T @ (P - Y) / n + self.l2 / n * self.W)
            self.b -= self.lr * (P - Y).mean(axis=0)
        return self

    def predict_proba(self, X):
        return softmax(X @ self.W + self.b)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

In [ ]:
X3, y3 = make_blobs(n_samples=450, centers=3, cluster_std=1.8, random_state=7)
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.25, random_state=0)

sm = SoftmaxRegression(lr=0.05, n_iters=2000).fit(X3_train, y3_train)
print("Test accuracy:", accuracy_score(y3_test, sm.predict(X3_test)))

plot_decision_boundary(sm, X3, y3, "Softmax regression — 3 classes")
plt.show()

## Summary
* Logistic regression outputs calibrated-ish probabilities and gives a **linear** decision boundary.
* The cross-entropy loss is convex → gradient descent finds the global optimum.
* Softmax regression extends it to many classes; non-linear boundaries need feature engineering (e.g. polynomial features).